# Pixel Ecosystem Classification with AlphaEarth Foundations Embeddings and Random Forest

[Google DeepMind AlphaEarth Foundations](https://deepmind.google/blog/alphaearth-foundations-helps-map-our-planet-in-unprecedented-detail/) is a geospatial embedding model from Google DeepMind. Rather than producing a satellite image, it produces a 64-dimensional "embedding" for every 10m pixel on Earth's land and coastal surface, for each year since 2017. Each embedding summarises information from many satellite and geophysical data sources (optical, radar, LiDAR, climate simulations, and more) into a single compact vector that captures the character of that location through for that year.

From Google DeepMind: "AlphaEarth Foundations provides a powerful new lens for understanding our planet by solving two major challenges: data overload and inconsistent information."

Because the embeddings encode a huge amount of information, they are very effective input features for simple machine learning classifiers - often more effective than raw satellite bands, and with far less data to load and process.

This notebook demonstrates how to:

* Load AlphaEarth Foundations embeddings as a cloud native Zarr dataset, hosted on [Source Cooperative](https://source.coop/tge-labs/aef-mosaic)
* Combine the embeddings with a point-based training dataset of ecosystem classes
* Train a Random Forest classifier and use it to map ecosystems across an area of interest

The training data is a set of labelled points over coastal and marine ecosystems in Nusa Tenggara Barat (West Nusa Tenggara), Indonesia, produced for the [Piksel-Ina notebooks](https://github.com/piksel-ina/piksel-notebooks). It labels points as Mangrove, Photic Coral Reef, Seagrass, Deep Water, Forest, Urban, or Sand.

This notebook follows the same workflow as the [Sentinel-2 machine learning example](https://github.com/auspatious/coastal-applications-workshop/blob/main/notebooks/examples/05_Sentinel-2_MachineLearning.ipynb) from the Coastal Applications Workshop, but swaps out Sentinel-2 bands (and indices) for AlphaEarth embeddings as the input features. This provides a richer, but less explicit dataset.

Here is a great visualisation of the AlphaEarth Foundations Embeddings: https://developmentseed.org/deck.gl-raster/examples/aef-mosaic/

## Set up

The first step is to import the required Python libraries.

* `xarray` is used to lazily open and slice the AlphaEarth embeddings mosaic stored as cloud-native Zarr, without loading the full global dataset into memory
* `odc-geo` (`odc.geo.xr`, `odc.geo.geom.Geometry`) provides CRS handling and efficient spatial cropping (`odc.geo.xr.crop`), plus RGB rendering and interactive map preview (`.odc.explore()`)
* `numpy` is used for array manipulation, masking nodata values, and de-quantizing the embeddings. Dask extends from basic numpy arrays.
* `geopandas` and `shapely` (`shapely.geometry.box`) are used to load, reproject, and explore the point training data, and to build the bounding box used to clip the embeddings to our area of interest
* `pandas` is used to assemble the extracted training features into a table for the classifier
* `scikit-learn` provides `PCA` (for visualizing the 64-band embeddings as RGB) and `RandomForestClassifier` (for land-cover classification)
* `matplotlib` and `branca` are used to build a categorical colormap and legend for the classified map
* `ipyleaflet` provides the basemap tiles used in interactive `.explore()` visualizations

In [ ]:
import folium
import geopandas as gpd
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import odc.geo.xr
import pandas as pd
import xarray as xr
from ipyleaflet import basemaps
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier

## 1. Load training data

The training data is a GeoJSON file of labelled points, hosted in the Piksel-Ina notebooks repository. Each point has a `Class` (a numeric code) and an `Ecosys_Typ` (a human readable ecosystem type).

We load it using `geopandas`.

In [ ]:
training_data_path = "./NusaTenggaraBarat_tdata.geojson"
training_points = gpd.read_file(training_data_path)

print(f"Loaded {len(training_points)} training points")
training_points.explore(column="Ecosys_Typ", legend=True, tiles=basemaps.Esri.WorldImagery, style_kwds={"radius": 3})

## 2. Define the area of interest

We use the bounding box of the training points as our area of interest.

In [ ]:
aoi_array = training_points.total_bounds
aoi_box = odc.geo.geom.box(*aoi_array, crs=training_points.crs)
aoi_box.explore()

## 3. Load AlphaEarth Foundations embeddings

The AlphaEarth Foundations mosaic is published on Source Cooperative by Taylor Geospatial as a cloud native [Zarr](https://zarr.dev/) store, accessed here through its `zarr.json` metadata document. Zarr is a chunked, compressed, cloud native array format which plays a similar role for multi-dimensional arrays that COGs (with STAC metadata) play for individual images.

Because it's cloud native, we can open the dataset lazily with `xarray` and only stream the chunks we need for our area of interest, without downloading the full global archive (which would be extremely slow and impractical).

See more about Zarr and cloud native geospatial formats here: https://guide.cloudnativegeo.org/zarr/intro.html

In [ ]:
embeddings = xr.open_zarr(
    "s3://us-west-2.opendata.source.coop/tge-labs/aef-mosaic/",
    storage_options={"anon": True},
    consolidated=False,
    chunks="auto",
    zarr_format=3,
)
# Note: this will take ~1 minute to initialise. No pixel data is downloaded yet, just the metadata.

### 3.1 Explore the dataset structure

Before subsetting, it's worth inspecting the dataset's dimensions, variables and attributes. AlphaEarth Foundations embeddings have 64 dimensions per pixel at 10m resolution (at the equator), with one composite per year.

Dimensions: Note the 9 times (years 2017-2025), the 64 bands (each embedding), and the y and x (the pixel coords).

In [ ]:
embeddings = embeddings.odc.assign_crs(training_points.crs) # Both datasets are in WGS84. Just being explicit.
embeddings

### 3.2 Subset to the area of interest and most recent year

This is a classic spatio-temporal filter. We use `odc.geo.xr.crop` to efficiently limit the global dataset to the area of interest. We also simply select the time `2025`.

After this filter note the differences in dimension to the ouptut above. We are now down to a single time of 2025, and ~2000x2000 10m pixels in the x and y dimensions, instead of the millions we had in the global dataset. The 64 embedding dimensions are still present.

In [ ]:
embeddings_clipped = (
    odc.geo.xr.crop(embeddings, aoi_box)
    .sel(time=2025)["embeddings"]
)

embeddings_clipped

Here we actually load the data so it is no longer lazy. This only loads the spatially and temporally filtered data, not the global dataset. This will take ~3 minutes (depending on bandwidth and AOI size).

In [ ]:
embeddings_clipped = embeddings_clipped.compute() # Load the lazy dask array.

### 3.3 Visualise the embeddings with a PCA false-colour composite

A 64-dimensional embedding can't be viewed directly as an image. To get around this we run Principal Component Analysis (PCA) across the 64 embedding dimension and map these three components to red, green and blue. Areas that look similar in this false-colour image have similar embeddings, and therefore similar land or seascape characteristics.

In [ ]:
# If you want, you can visualise the first 3 embeddings. This is what is done in the example linked at the top of this notebook.
# embeddings_ds = embeddings_clipped.sel(band=["A00", "A01", "A02"]).to_dataset(dim="band")
# embeddings_ds.odc.to_rgba(bands=["A00", "A01", "A02"], vmin=-127, vmax=127).odc.explore(tiles=basemaps.Esri.WorldImagery)

In [ ]:
# Stack spatial dims for PCA (band must be last axis for sklearn)
# Just think of this as reshaping the data for the PCA. We are not changing the data, just how it is represented in memory.
embeddings_stacked = embeddings_clipped.stack(pixel=("y", "x")).transpose("pixel", "band")
embeddings_stacked

In [ ]:
raw = embeddings_stacked.values  # int8, shape (pixel, band)
nodata = -128
valid_mask = ~(raw == nodata).all(axis=1)  # rows where not all bands are nodata

# De-quantize to float32 - PCA needs this (unlike RF)
data_float = ((raw.astype("float32") / 127.5) ** 2) * np.sign(raw)

pca = PCA(n_components=3)
pca_values_valid = pca.fit_transform(data_float[valid_mask])

# Rebuild full-length array, filling masked-out rows with NaN
pca_values = np.full((raw.shape[0], 3), np.nan, dtype="float32")
pca_values[valid_mask] = pca_values_valid

pca_da = xr.DataArray(
    pca_values,
    coords={"pixel": embeddings_stacked.pixel, "component": ["r", "g", "b"]},
    dims=["pixel", "component"],
).unstack("pixel")

pca_da = (pca_da - pca_da.min(["x", "y"])) / (pca_da.max(["x", "y"]) - pca_da.min(["x", "y"]))
pca_da = pca_da.odc.assign_crs(embeddings_clipped.odc.crs)

Note that the colors in this PCA visualisation do not yet map to any ecosystem type, they just show us similarities and differences in this area. Note the clear sea, land, vegetation areas. Scroll up to our training data to imagine how these colours could represent the classes in the training data.

In [ ]:
pca_ds = pca_da.to_dataset(dim="component")
pca_ds.odc.to_rgba(bands=["r", "g", "b"]).odc.explore(tiles=basemaps.Esri.WorldImagery)

## 4. Prepare the training array

Now we extract the embedding values at each training point location, and join them with the point's `Ecosys_Typ` label. This produces a table where each row is a training point, the first column is the ecosystem type, and the remaining columns are the 64 embedding dimensions.

In [ ]:
# Build an xarray Dataset of just x/y coords to drive vectorized nearest-neighbor selection
training_xr = training_points.assign(
    x=training_points.geometry.x, y=training_points.geometry.y
)[["x", "y"]].to_xarray()

training_values = (
    embeddings_clipped
    .sel(training_xr, method="nearest", tolerance=0.001)
    .squeeze()
    .transpose(..., "band")
    .to_pandas()
)

training_array = pd.concat(
    [training_points["Ecosys_Typ"].reset_index(drop=True), training_values.reset_index(drop=True)],
    axis=1,
)
training_array = training_array.drop(columns=["x", "y"], errors="ignore")
training_array = training_array[~(training_array.drop(columns=["Ecosys_Typ"]) == nodata).all(axis=1)]
training_array = training_array.dropna()
training_array.head()

## 5. Train a Random Forest classifier

As in the Sentinel-2 machine learning example, we pass simple numpy arrays into a Random Forest classifier: one array of observed embedding values, and one array of class labels.

Rather than hand-crafting our own decision logic for how 64 embedding values map to an ecosystem type, a Random Forest builds many decision trees, each on a different random subset of the data, and averages their predictions. This reduces overfitting and improves accuracy compared to any single decision tree.

Learn more about Random Forests in the [scikit-learn documentation](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html).

In [ ]:
classes = training_array["Ecosys_Typ"].values
observations = training_array.drop(columns=["Ecosys_Typ"]).values  # raw int8 (quantized) embeddings

classifier = RandomForestClassifier(class_weight="balanced", n_estimators=200, n_jobs=-1)
model = classifier.fit(observations, classes)

## 6. Predict across the area of interest

To classify every pixel, we reshape the embedding array into a long table of observations (one row per pixel), predict a class for each, then reshape the result back into a 2D image.

In [ ]:
predicted = np.full(raw.shape[0], "nodata", dtype=object)
predicted[valid_mask] = model.predict(raw[valid_mask])

predicted_da = xr.DataArray(
    predicted.reshape(len(embeddings_clipped.y), len(embeddings_clipped.x)),
    coords={"y": embeddings_clipped.y, "x": embeddings_clipped.x},
    dims=["y", "x"],
).odc.assign_crs(embeddings_clipped.odc.crs)

predicted_da

## 7. Visualise the results

Finally, we bring together the PCA false-colour composite, the predicted classes, and the original training points on a single interactive map.

In [ ]:
from folium.map import CustomPane

class_names = sorted(set(predicted_da.values[predicted_da.values != None].flatten().tolist()))
class_to_int = {name: i for i, name in enumerate(class_names)}
n_classes = len(class_names)

predicted_int = np.vectorize(lambda x: class_to_int.get(x, -1))(predicted_da.values).astype("int8")
predicted_int_da = xr.DataArray(
    predicted_int,
    coords={"y": predicted_da.y, "x": predicted_da.x},
    dims=["y", "x"],
).odc.assign_crs(predicted_da.odc.crs)

cmap = plt.get_cmap("tab20", n_classes)
class_colors = {name: mcolors.rgb2hex(cmap(i)) for i, name in enumerate(class_names)}

m = folium.Map(
    location=[float(predicted_da.y.mean()), float(predicted_da.x.mean())],
    zoom_start=11,
    tiles=basemaps.Esri.WorldImagery,
)

points_pane = CustomPane("points_pane", z_index=627, pointer_events=True)
predicted_pane = CustomPane("predicted_pane", z_index=626)
points_pane.add_to(m)
predicted_pane.add_to(m)

points_group = folium.FeatureGroup(name="Training Points")
for _, row in training_points.to_crs("EPSG:4326").iterrows():
    label = row["Ecosys_Typ"]
    marker = folium.CircleMarker(
        location=(row.geometry.y, row.geometry.x),
        radius=4,
        color="black",
        weight=1,
        fill=True,
        fill_color=class_colors.get(label, "gray"),
        fill_opacity=0.9,
        popup=label,
    )
    marker.options["pane"] = "points_pane"
    marker.add_to(points_group)
points_group.add_to(m)

predicted_int_da.odc.explore(
    map=m,
    cmap=cmap,
    vmin=0,
    vmax=n_classes - 1,
    name="Predicted Ecosystem Type",
    layer_control=False,
    pane="predicted_pane",
)

pca_ds.odc.to_rgba(bands=["r", "g", "b"]).odc.explore(
    map=m,
    name="PCA (RGB)",
    layer_control=False,
)

legend_rows = "".join(
    f'<div style="display:flex;align-items:center;margin-bottom:2px;">'
    f'<span style="background:{class_colors[name]};width:14px;height:14px;'
    f'display:inline-block;margin-right:6px;border:1px solid #333;"></span>{name}</div>'
    for name in class_names
)
legend_html = f"""
<div style="
    position: fixed; bottom: 30px; left: 30px; z-index: 9999;
    background: white; padding: 10px 12px; border: 1px solid #999;
    border-radius: 4px; font-size: 13px; box-shadow: 2px 2px 6px rgba(0,0,0,0.3);">
    <b>Predicted Ecosystem Type</b><br>{legend_rows}
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

folium.LayerControl(collapsed=False).add_to(m)
m

## Conclusion and next steps

This notebook showed how AlphaEarth Foundations embeddings, streamed directly from a cloud native Zarr store on Source Cooperative, can be combined with a small set of labelled training data points to classify pixel-level ecosystems - without ever needing to load or process a raw satellite image.

As a next step, you may like to:

1. Try a different area of interest and/or use your own training points
2. Compare classification accuracy against the Sentinel-2-based approach used in the [coastal applications workshop](https://github.com/auspatious/coastal-applications-workshop)
3. Explore how classification accuracy changes across different years of AlphaEarth embeddings
4. Do a change analysis. Run this for 2017 and for 2025 and then analyse how the area of interest has changed. You could use [this](https://knowledge.dea.ga.gov.au/notebooks/How_to_guides/Land_cover_change_mapping/) as a base.